# Quantization Efficiency Analysis — FP32 vs Runtime Dynamic INT8

This notebook is a **standalone efficiency benchmark** built to match the project's existing `quantization.ipynb`.

It does **not train** a model and does **not require a saved INT8 checkpoint**.

Actual project pipeline reproduced here:

```text
models/fp32/seed<SEED>/final
          │
          ├─────────────── FP32 benchmark
          │
          └─ deepcopy → TorchAO quantize_
                         Int8DynamicActivationInt8WeightConfig
                                  │
                                  └─ INT8 benchmark
```

## Primary metrics

1. Serialized model-state size
2. Registered parameter/buffer tensor memory
3. Peak inference memory
4. Inference latency
5. Throughput
6. Runtime process/device memory
7. Full HANS inference time

The notebook also records **runtime quantization setup time** as a supplementary measurement.

---

## 🔴 CELLS YOU MAY NEED TO CHANGE

Only cells explicitly headed **🔴 USER CONFIGURATION** should normally require editing.

For the current project structure, the defaults already match:

- FP32 models: `../models/fp32/seed<SEED>/final`
- HANS: official raw HANS TSV used by `quantization.ipynb`
- Results: `../results/efficiency`
- `MAX_LENGTH = 128`
- `BATCH_SIZE = 32`

Start with seed 42. After it works, change only `SEED` to 123 and 456.

**Measurement caution:** TorchAO logical tensor byte counts and single-process RSS are reported as diagnostics, not evidence of isolated INT8 model memory. Runtime benchmarks use repeated passes.

## 1. Imports

In [ ]:
from pathlib import Path
import copy
import gc
import io
import json
import os
import platform
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchao

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from torchao.quantization import (
    quantize_,
    Int8DynamicActivationInt8WeightConfig,
)

try:
    import psutil
except ImportError:
    psutil = None

print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)
print("Platform:", platform.platform())
print("CUDA available:", torch.cuda.is_available())
print(
    "MPS available:",
    hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
)

## 🔴 2. USER CONFIGURATION — CHECK THIS CELL

**This is the main cell you change.**

For your existing directory structure, normally change only:

```python
SEED = 42
```

to `123` or `456` for the other runs.

If this notebook sits in the same notebook directory as your existing quantization notebook, the paths below match your project.

In [ ]:
# ============================================================
# 🔴 USER CONFIGURATION
# ============================================================

SEED = 42

MAX_LENGTH = 128
BATCH_SIZE = 32

# Number of untimed warm-up batches before measuring latency.
WARMUP_BATCHES = 20

# Timed complete-HANS passes per model (3 recommended; use 1 for debugging).
BENCHMARK_REPEATS = 3

# None = benchmark all 30,000 HANS examples.
# For a quick debugging run, temporarily use e.g. 1000.
MAX_EXAMPLES = None

# Existing project model layout:
MODEL_PATH = Path(f"../models/fp32/seed{SEED}/final")

# Benchmark outputs:
RESULTS_DIR = Path("../results/efficiency")

# Same official HANS source used in your quantization.ipynb.
HANS_URL = (
    "https://raw.githubusercontent.com/tommccoy1/hans/"
    "master/heuristics_evaluation_set.txt"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Seed:", SEED)
print("Model path:", MODEL_PATH)
print("Results directory:", RESULTS_DIR)
print("Batch size:", BATCH_SIZE)
print("Max length:", MAX_LENGTH)

## 3. Validate paths

If this cell fails with `Model directory not found`, the notebook is being run from a different working directory than your original notebooks.

In that case, change **only `MODEL_PATH` in the USER CONFIGURATION cell above**.

In [ ]:
print("Current working directory:", Path.cwd())
print("Resolved model path:", MODEL_PATH.resolve())

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model directory not found: {MODEL_PATH.resolve()}\n"
        "Update MODEL_PATH in the USER CONFIGURATION cell."
    )

print("✓ FP32 model directory found")

## 4. Benchmark device

The benchmark uses one device/backend for **both FP32 and INT8**.

TorchAO dynamic INT8 kernel support is backend-dependent. This cell chooses CUDA when available, then MPS, otherwise CPU.

If the INT8 model cannot execute on the selected accelerator in your installed TorchAO/PyTorch environment, set `DEVICE = torch.device("cpu")` here and benchmark **both models on CPU**. Do not compare FP32 on one backend with INT8 on another as a quantization speedup.

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Benchmark device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 5. Load FP32 model and tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model_fp32 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)
model_fp32.eval()

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

print("✓ FP32 model loaded")
print("Labels:", model_fp32.config.id2label)

## 6. Create runtime INT8 model

This is intentionally the same procedure used by your existing `quantization.ipynb`.

The quantized model is created in memory and is **not saved as a checkpoint**.

In [ ]:
quantization_start = time.perf_counter()

model_int8 = copy.deepcopy(model_fp32)

quantize_(
    model_int8,
    Int8DynamicActivationInt8WeightConfig()
)

model_int8.eval()

quantization_seconds = time.perf_counter() - quantization_start

print("✓ Dynamic INT8 quantization completed")
print(f"Runtime quantization setup time: {quantization_seconds:.3f} seconds")

## 7. Load HANS — same method as your quantization notebook

This deliberately **does not use**:

```python
load_dataset("hans")
```

because newer `datasets` versions reject the legacy `hans.py` dataset script.

Instead, it uses the same official raw HANS TSV URL already used successfully in your `quantization.ipynb`.

In [ ]:
hans_dataset = load_dataset(
    "csv",
    data_files={"validation": HANS_URL},
    delimiter="\t",
)

hans = hans_dataset["validation"]

hans_eval = hans.rename_columns({
    "sentence1": "premise",
    "sentence2": "hypothesis",
})

if MAX_EXAMPLES is not None:
    hans_eval = hans_eval.select(
        range(min(MAX_EXAMPLES, len(hans_eval)))
    )

print("HANS examples:", len(hans_eval))
print("First premise:", hans_eval[0]["premise"])
print("First hypothesis:", hans_eval[0]["hypothesis"])

## 8. Tokenize HANS and build DataLoader

This matches your existing settings:

- tokenizer from the fine-tuned FP32 checkpoint
- truncation
- `MAX_LENGTH = 128`
- dynamic padding
- `BATCH_SIZE = 32`
- no shuffle

Labels and HANS metadata are not needed for an efficiency benchmark.

In [ ]:
def tokenize_hans(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

hans_tokenized = hans_eval.map(
    tokenize_hans,
    batched=True,
)

# BERT normally has token_type_ids. Build the list defensively.
model_columns = [
    col
    for col in ["input_ids", "token_type_ids", "attention_mask"]
    if col in hans_tokenized.column_names
]

hans_model_inputs = hans_tokenized.select_columns(model_columns)

hans_loader = DataLoader(
    hans_model_inputs,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)

print("Model input columns:", model_columns)
print(f"Examples: {len(hans_model_inputs):,}")
print(f"Batches: {len(hans_loader):,}")

expected_batches = int(np.ceil(len(hans_model_inputs) / BATCH_SIZE))
assert len(hans_loader) == expected_batches

if MAX_EXAMPLES is None:
    assert len(hans_model_inputs) == 30000, (
        f"Expected 30,000 HANS examples, got {len(hans_model_inputs):,}"
    )

print("✓ HANS benchmark workload ready")

# Metric 1 — Serialized model-state size

Because INT8 exists only at runtime, there is no INT8 model directory to compare against.

For a fair FP32-vs-INT8 measurement, this cell serializes each model's `state_dict()` to an **in-memory buffer** using the same method and measures the number of bytes.

Nothing is written to disk.

In [ ]:
def serialized_state_size_mib(model):
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    nbytes = buffer.tell()
    buffer.close()
    return nbytes / (1024 ** 2)

fp32_serialized_mib = serialized_state_size_mib(model_fp32)
int8_serialized_mib = serialized_state_size_mib(model_int8)

storage_df = pd.DataFrame([
    {"model": "FP32", "serialized_state_MiB": fp32_serialized_mib},
    {"model": "INT8", "serialized_state_MiB": int8_serialized_mib},
])

display(storage_df)

storage_reduction = 100 * (
    1 - int8_serialized_mib / fp32_serialized_mib
)
storage_ratio = fp32_serialized_mib / int8_serialized_mib

print(f"Compression ratio: {storage_ratio:.3f}x")
print(f"Serialized-state reduction: {storage_reduction:.2f}%")

# Metric 2 — Logical registered tensor memory (diagnostic only)

TorchAO uses tensor subclasses; `numel() * element_size()` can count **logical** bytes instead of packed quantized storage. A result such as **417.66 MiB for both models** does **not** mean that INT8 failed to compress weights. **Do not cite this metric as actual INT8 model-memory savings.** Use the serialized state size above for disk footprint; report this metric only with the limitation attached.

In [ ]:
def tensor_memory_stats(model):
    parameter_bytes = sum(
        p.numel() * p.element_size()
        for p in model.parameters()
    )
    buffer_bytes = sum(
        b.numel() * b.element_size()
        for b in model.buffers()
    )

    return {
        "parameter_MiB": parameter_bytes / (1024 ** 2),
        "buffer_MiB": buffer_bytes / (1024 ** 2),
        "registered_tensor_MiB":
            (parameter_bytes + buffer_bytes) / (1024 ** 2),
    }

tensor_memory_df = pd.DataFrame([
    {"model": "FP32", **tensor_memory_stats(model_fp32)},
    {"model": "INT8", **tensor_memory_stats(model_int8)},
])

display(tensor_memory_df)

# Metrics 3–7 — Inference benchmark

The same HANS workload and device are used for both models. Timing uses warm-up and **repeated full inference passes**, summarized using the median. Order alternates across repeats (FP32→INT8, then INT8→FP32) to reduce systematic order effects.

**Important memory caveat:** Both model objects coexist in a single notebook process. RSS includes Python, cached allocator memory, DataLoader objects, and the *other* model. The RSS metric is a **process-level diagnostic, not isolated model inference memory**. Use separate fresh processes to claim an attributable FP32-vs-INT8 process-memory difference.

Peak CUDA allocator memory is device allocation within the process, not exclusively the active model; it can also include both models. The inference-loop time includes DataLoader batching and transfers, but excludes initial HANS loading/tokenization and quantization setup.

In [ ]:
def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        try:
            torch.mps.synchronize()
        except Exception:
            pass


def process_rss_mib():
    if psutil is None:
        return np.nan

    return (
        psutil.Process(os.getpid()).memory_info().rss
        / (1024 ** 2)
    )


def device_memory_allocated_mib(device):
    if device.type == "cuda":
        return (
            torch.cuda.memory_allocated(device)
            / (1024 ** 2)
        )

    if device.type == "mps":
        try:
            return (
                torch.mps.current_allocated_memory()
                / (1024 ** 2)
            )
        except Exception:
            return np.nan

    return np.nan


def reset_memory_tracking(device):
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)

    elif device.type == "mps":
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


def allocator_peak_mib(device):
    if device.type == "cuda":
        return (
            torch.cuda.max_memory_allocated(device)
            / (1024 ** 2)
        )

    # No directly equivalent peak allocator API is assumed for MPS.
    return np.nan

In [ ]:
def benchmark_model(
    model,
    dataloader,
    device,
    model_name,
    warmup_batches=20,
):
    # Move model first, then clear/reset measurement state.
    model = model.to(device)
    model.eval()

    reset_memory_tracking(device)

    # -------------------------
    # Warm-up
    # -------------------------
    with torch.inference_mode():
        for batch_idx, batch in enumerate(dataloader):
            if batch_idx >= warmup_batches:
                break

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            _ = model(**batch)

    synchronize(device)

    # Warm-up allocations should not count toward CUDA peak.
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    # -------------------------
    # Timed full inference pass
    # -------------------------
    rss_before = process_rss_mib()
    device_before = device_memory_allocated_mib(device)

    sampled_rss = [rss_before]
    sampled_device_memory = [device_before]

    batch_latencies_s = []
    batch_sizes = []

    total_examples = 0

    synchronize(device)
    full_start = time.perf_counter()

    with torch.inference_mode():
        for batch in dataloader:
            current_batch_size = batch["input_ids"].shape[0]

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            synchronize(device)
            batch_start = time.perf_counter()

            _ = model(**batch)

            synchronize(device)
            batch_elapsed = time.perf_counter() - batch_start

            batch_latencies_s.append(batch_elapsed)
            batch_sizes.append(current_batch_size)
            total_examples += current_batch_size

            sampled_rss.append(process_rss_mib())
            sampled_device_memory.append(
                device_memory_allocated_mib(device)
            )

    synchronize(device)
    end_to_end_seconds = time.perf_counter() - full_start

    rss_after = process_rss_mib()
    device_after = device_memory_allocated_mib(device)

    valid_rss = [
        x for x in sampled_rss
        if np.isfinite(x)
    ]
    valid_device = [
        x for x in sampled_device_memory
        if np.isfinite(x)
    ]

    batch_latencies_ms = (
        np.asarray(batch_latencies_s, dtype=float) * 1000
    )
    batch_sizes_arr = np.asarray(batch_sizes, dtype=float)

    example_latencies_ms = (
        batch_latencies_ms / batch_sizes_arr
    )

    result = {
        "model": model_name,
        "device": str(device),
        "examples": int(total_examples),
        "batches": len(batch_latencies_s),

        # Metric 7
        "end_to_end_seconds": end_to_end_seconds,

        # Metric 5
        "throughput_examples_per_sec":
            total_examples / end_to_end_seconds,

        # Metric 4
        "mean_batch_latency_ms":
            float(batch_latencies_ms.mean()),
        "median_batch_latency_ms":
            float(np.median(batch_latencies_ms)),
        "p95_batch_latency_ms":
            float(np.percentile(batch_latencies_ms, 95)),
        "mean_example_latency_ms":
            float(example_latencies_ms.mean()),
        "median_example_latency_ms":
            float(np.median(example_latencies_ms)),

        # Metrics 3 / 6
        "process_rss_before_MiB": rss_before,
        "process_rss_after_MiB": rss_after,
        "peak_sampled_process_rss_MiB":
            max(valid_rss) if valid_rss else np.nan,

        "device_memory_before_MiB": device_before,
        "device_memory_after_MiB": device_after,
        "peak_sampled_device_memory_MiB":
            max(valid_device) if valid_device else np.nan,

        "allocator_peak_device_memory_MiB":
            allocator_peak_mib(device),
    }

    # Return model to CPU before benchmarking the next model.
    model.to("cpu")
    synchronize(device)
    reset_memory_tracking(device)

    return result, batch_latencies_ms

## 9–11. Repeated paired benchmark

Runs both models on the same backend and data for `BENCHMARK_REPEATS` complete passes. The execution order alternates. The summary uses **median** values across passes, and the raw per-run results and batch latencies are saved later to CSV.

In [ ]:
all_runtime_runs = []
all_latency_rows = []

for repeat in range(1, BENCHMARK_REPEATS + 1):
    order = [("FP32", model_fp32), ("INT8", model_int8)]
    if repeat % 2 == 0:
        order.reverse()

    for model_name, current_model in order:
        print(f"Running repeat {repeat}/{BENCHMARK_REPEATS}: {model_name}")
        result, batch_latencies_ms = benchmark_model(
            model=current_model,
            dataloader=hans_loader,
            device=DEVICE,
            model_name=model_name,
            warmup_batches=WARMUP_BATCHES,
        )
        result["repeat"] = repeat
        result["execution_order"] = order.index((model_name, current_model)) + 1
        all_runtime_runs.append(result)

        for batch_index, latency_ms in enumerate(batch_latencies_ms, start=1):
            all_latency_rows.append({
                "seed": SEED, "model": model_name, "repeat": repeat,
                "batch_index": batch_index, "latency_ms": float(latency_ms),
            })

runtime_runs_df = pd.DataFrame(all_runtime_runs)
batch_latencies_df = pd.DataFrame(all_latency_rows)

# Aggregate only numeric measurements. Exclude bookkeeping columns.
metric_columns = [
    col for col in runtime_runs_df.select_dtypes(include=["number"]).columns
    if col not in ("repeat", "execution_order")
]
runtime_df = (
    runtime_runs_df.groupby("model", as_index=False)[metric_columns]
    .median(numeric_only=True)
)
runtime_df["device"] = str(DEVICE)
runtime_df["repeats"] = BENCHMARK_REPEATS

# Backwards-compatible named variables for the plotting cell.
fp32_batch_latencies_ms = batch_latencies_df.loc[
    batch_latencies_df.model == "FP32", "latency_ms"
].to_numpy()
int8_batch_latencies_ms = batch_latencies_df.loc[
    batch_latencies_df.model == "INT8", "latency_ms"
].to_numpy()

display(runtime_runs_df)
display(runtime_df)

## 12. Final seven-metric table

For peak inference memory:

1. CUDA allocator peak is preferred when available.
2. Otherwise sampled device memory is used.
3. Otherwise sampled process RSS is used.

The table explicitly records which memory source was used.

In [ ]:
def row_value(df, model_name, column):
    return float(
        df.loc[df["model"] == model_name, column].iloc[0]
    )


def percent_reduction(fp32, int8):
    if not (
        np.isfinite(fp32)
        and np.isfinite(int8)
        and fp32 != 0
    ):
        return np.nan

    return 100 * (1 - int8 / fp32)


# Determine best available peak-memory metric.
memory_candidates = [
    (
        "allocator_peak_device_memory_MiB",
        "device allocator peak",
    ),
    (
        "peak_sampled_device_memory_MiB",
        "sampled device memory",
    ),
    (
        "peak_sampled_process_rss_MiB",
        "sampled process RSS",
    ),
]

peak_column = None
peak_source = None

for candidate, source in memory_candidates:
    fp = row_value(runtime_df, "FP32", candidate)
    q = row_value(runtime_df, "INT8", candidate)

    if np.isfinite(fp) and np.isfinite(q):
        peak_column = candidate
        peak_source = source
        break

if peak_column is None:
    raise RuntimeError(
        "No usable runtime memory measurement was available."
    )


final_rows = []

# 1
fp = row_value(storage_df, "FP32", "serialized_state_MiB")
q = row_value(storage_df, "INT8", "serialized_state_MiB")
final_rows.append({
    "metric": "1. Serialized model-state size",
    "FP32": fp,
    "INT8": q,
    "unit": "MiB",
    "comparison":
        f"{percent_reduction(fp, q):.2f}% reduction",
})

# 2
fp = row_value(
    tensor_memory_df, "FP32", "registered_tensor_MiB"
)
q = row_value(
    tensor_memory_df, "INT8", "registered_tensor_MiB"
)
final_rows.append({
    "metric": "2. Registered tensor memory",
    "FP32": fp,
    "INT8": q,
    "unit": "MiB",
    "comparison":
        "diagnostic only — packed TorchAO storage not measured",
})

# 3
fp = row_value(runtime_df, "FP32", peak_column)
q = row_value(runtime_df, "INT8", peak_column)
final_rows.append({
    "metric": f"3. Peak inference memory ({peak_source})",
    "FP32": fp,
    "INT8": q,
    "unit": "MiB",
    "comparison": "confounded by co-resident models (diagnostic only)",
})

# 4
fp = row_value(
    runtime_df, "FP32", "median_batch_latency_ms"
)
q = row_value(
    runtime_df, "INT8", "median_batch_latency_ms"
)
final_rows.append({
    "metric": "4. Median batch latency",
    "FP32": fp,
    "INT8": q,
    "unit": "ms/batch",
    "comparison": f"{fp / q:.3f}x speedup",
})

# 5
fp = row_value(
    runtime_df, "FP32", "throughput_examples_per_sec"
)
q = row_value(
    runtime_df, "INT8", "throughput_examples_per_sec"
)
final_rows.append({
    "metric": "5. Throughput",
    "FP32": fp,
    "INT8": q,
    "unit": "examples/s",
    "comparison":
        f"{100 * (q / fp - 1):+.2f}% change",
})

# 6 — process memory specifically
fp = row_value(
    runtime_df, "FP32", "peak_sampled_process_rss_MiB"
)
q = row_value(
    runtime_df, "INT8", "peak_sampled_process_rss_MiB"
)
final_rows.append({
    "metric": "6. Peak process runtime memory",
    "FP32": fp,
    "INT8": q,
    "unit": "MiB",
    "comparison":
        (
            "not isolated — diagnostic only"
        ),
})

# 7
fp = row_value(
    runtime_df, "FP32", "end_to_end_seconds"
)
q = row_value(
    runtime_df, "INT8", "end_to_end_seconds"
)
final_rows.append({
    "metric": "7. Full HANS inference time",
    "FP32": fp,
    "INT8": q,
    "unit": "seconds",
    "comparison": f"{fp / q:.3f}x speedup",
})

final_table = pd.DataFrame(final_rows)

display(final_table)

print(
    f"Supplementary runtime quantization setup time: "
    f"{quantization_seconds:.3f} seconds"
)

## 13. Batch latency distribution across all repeats

In [ ]:
latency_plot_df = pd.DataFrame({
    "FP32": pd.Series(fp32_batch_latencies_ms),
    "INT8": pd.Series(int8_batch_latencies_ms),
})

ax = latency_plot_df.boxplot(
    figsize=(7, 5),
    showfliers=False,
)

ax.set_ylabel("Batch latency (ms)")
ax.set_title(f"FP32 vs INT8 HANS batch latency — seed {SEED}")

plt.tight_layout()
plt.show()

## 14. Save reusable CSV results

The exports include the seven-metric summary, **per-repeat runtime measurements**, **per-batch latency observations**, the serialized size, diagnostic tensor memory, and environment metadata. The notebook also maintains a combined `all_seeds_efficiency_summary.csv` table that updates the current seed instead of adding duplicate rows when you rerun it.

**Interpretation:** serialized footprint and repeated inference performance can be compared directly. Registered-tensor byte estimates, peak allocator memory, and process RSS are **diagnostics** and should not be claimed as isolated quantized-model memory reductions.

In [ ]:
environment = {
    "seed": SEED,
    "model_path": str(MODEL_PATH.resolve()),
    "device": str(DEVICE),
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "torchao_version": torchao.__version__,
    "cuda_version": torch.version.cuda,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "warmup_batches": WARMUP_BATCHES,
    "benchmark_repeats": BENCHMARK_REPEATS,
    "num_hans_examples": len(hans_model_inputs),
    "runtime_quantization_setup_seconds": quantization_seconds,
    "memory_warning": "RSS/allocator metrics measured in a shared notebook process with both models resident; not isolated model-memory measurements",
    "tensor_memory_warning": "TorchAO tensor subclass numel*element_size is a logical estimate, not necessarily packed storage",
}
if DEVICE.type == "cuda":
    environment["gpu_name"] = torch.cuda.get_device_name(0)

# One result per seed/model, separate from detailed per-repeat data.
summary = final_table.copy()
summary.insert(0, "seed", SEED)
summary.insert(1, "device", str(DEVICE))
summary["benchmark_repeats"] = BENCHMARK_REPEATS
summary["model_path"] = str(MODEL_PATH.resolve())

exports = {
    f"seed{SEED}_efficiency_summary.csv": summary,
    f"seed{SEED}_runtime_median.csv": runtime_df,
    f"seed{SEED}_runtime_repeats.csv": runtime_runs_df,
    f"seed{SEED}_batch_latency.csv": batch_latencies_df,
    f"seed{SEED}_storage.csv": storage_df,
    f"seed{SEED}_tensor_memory_diagnostic.csv": tensor_memory_df,
}
for filename, frame in exports.items():
    frame.to_csv(RESULTS_DIR / filename, index=False)

# Convenient wide format: one row per seed, easily usable for analysis/plots.
wide_row = {"seed": SEED, "device": str(DEVICE),
            "benchmark_repeats": BENCHMARK_REPEATS,
            "quantization_setup_seconds": quantization_seconds,
            "memory_metrics_isolated": False}
for _, row in final_table.iterrows():
    index = int(str(row["metric"]).split(".")[0])
    wide_row[f"metric{index}_name"] = row["metric"]
    wide_row[f"metric{index}_fp32"] = row["FP32"]
    wide_row[f"metric{index}_int8"] = row["INT8"]
    wide_row[f"metric{index}_unit"] = row["unit"]
    wide_row[f"metric{index}_comparison"] = row["comparison"]

wide_path = RESULTS_DIR / "all_seeds_efficiency_summary.csv"
new_row = pd.DataFrame([wide_row])
if wide_path.exists():
    existing = pd.read_csv(wide_path)
    existing = existing.loc[existing["seed"] != SEED]
    combined = pd.concat([existing, new_row], ignore_index=True)
else:
    combined = new_row
combined.sort_values("seed").to_csv(wide_path, index=False)

with open(RESULTS_DIR / f"seed{SEED}_environment.json", "w") as f:
    json.dump(environment, f, indent=2)

print("Saved to:", RESULTS_DIR.resolve())
for path in exports:
    print("  ", path)
print("   all_seeds_efficiency_summary.csv")
display(combined)

# What you change for the other seeds

After seed 42 completes successfully, return to the **🔴 USER CONFIGURATION** cell and change only:

```python
SEED = 123
```

Run all cells again.

Then:

```python
SEED = 456
```

and run all cells again.

Because `MODEL_PATH` is constructed from `SEED`, it automatically becomes:

```text
../models/fp32/seed123/final
../models/fp32/seed456/final
```

The output filenames also change automatically.

## Normally, do NOT change

- `MAX_LENGTH = 128`
- `BATCH_SIZE = 32`
- HANS URL
- quantization configuration
- tokenizer
- warm-up methodology
- DataLoader settings

Changing those between FP32 and INT8 would weaken the comparison.